Run the setup cell once, then run numbered cells individually in order. Edit CONFIG before selecting the physical camera. Close CamWare and the camera GUI before camera sections. Each section closes its camera handle automatically and writes a JSON report. Acquired HDF5 files are retained on CONFIG.output_dir. Section 8 is optional and deferred by default.

Choose the project's .venv Python interpreter as the notebook kernel. If notebook support is unavailable, run one section from PowerShell: `.\.venv\Scripts\python.exe scripts/lab_computer_acceptance.py --section 3 --source mock`. Use `--source physical` only after reviewing the settings in the Python file.

Run one cell at a time; avoid Run All. The long reliability section can write about 74 GB at default settings. No hardware acceptance is certified from mock results.

In [ ]:
from pathlib import Path
import sys

# Start the notebook in the repository root or scripts/ directory.
start = Path.cwd()
ROOT = next((p for p in (start, *start.parents) if (p / "src/hhg_control").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the LabSoftware - Codex checkout")
sys.path.insert(0, str(ROOT))
from scripts import lab_computer_acceptance as lab

CONFIG = lab.Settings()
# Edit these before running the remaining cells:
CONFIG.source = "mock"  # Set to "physical" on the main PC after reviewing settings.
CONFIG.output_dir = ROOT / "data/test_data/lab_acceptance"  # e.g. Path("D:/CameraTests")
CONFIG.expected_serial = None  # Fill from physical camera label.
CONFIG.usb_port_note = "FILL IN: USB controller, port and cable"
CONFIG.roi = (612, 1016, 2352, 1144)
CONFIG.exposure_s = 0.001
CONFIG.readout = lab.ReadoutMode.GLOBAL_SHUTTER
CONFIG.target_fps = 455.0  # Storage load/reference; replace with measured rate.
CONFIG.camware_fps = None  # Matched measured CamWare capture baseline.
CONFIG.sustained_seconds = 300.0
CONFIG.max_section_raw_gb = 100.0
CONFIG.trigger_bench_ready = False  # Section 8 remains deferred.
print(CONFIG)


%% 1 — Readiness and storage (no physical camera is opened)
1. Set output_dir to the FINAL acquisition drive; fill usb_port_note.
2. Run this section with the final interpreter and checkout. Review pytest log.
3. Repeat writer measurements after warm-up and with normal background apps.
4. Watch Task Manager > Performance > Disk during a longer write. Short results
   can fit in OS/device caches; fsync timings still depend on the storage stack.
Benchmarks: all offline tests pass; sufficient free space; proposed unpaced
capacity >= 1.30 * (ROI width * height * 2 bytes * required fps). At 1740x128,
455 fps means 202.68 MB/s raw and a proposed 263.48 MB/s writer-capacity target.
Paced replay: every ID/image marker must agree, no error; review max lateness.
Replay waits for scheduled delivery and cannot prove physical FIFO tolerance.


In [ ]:
REPORT_1 = lab.execute(1, CONFIG)


%% 2 — Physical identity, settings and reconnect
1. Close CamWare/GUI. Review exposure_s, roi, readout; set source="physical".
2. Fill expected_serial from the camera label; execute this cell.
3. In GUI separately check Rolling -> Global -> Rolling -> intended mode.
   Camera reboot may take seconds. Confirm final ROI/exposure/trigger readback.
4. Close/reopen the GUI; repeat once. Do not run GUI and script simultaneously.
Benchmarks: exact serial/ROI/modes; exposure agrees to SDK precision; two
script reconnects agree; GUI has no stuck controls/crash and shows final mode.


In [ ]:
REPORT_2 = lab.execute(2, CONFIG)


%% 3 — Short acquisition: 20 frames, then 1,000
1. Complete section 2 first, then run this cell.
2. Inspect first/middle/last images in the GUI/analysis viewer.
3. Check actual scene, sensor orientation and intended ROI coverage.
Benchmarks: complete files, exact counts, uint16 [frame,y,x], all IDs
consecutive, no recorder/FIFO/writer errors. Appearance must match the scene;
dark/light statistics are observations, not a calibrated noise specification.


In [ ]:
REPORT_3 = lab.execute(3, CONFIG)


%% 4 — Camera download vs saving speed (3 x 10,000 by default)
1. In CamWare measure >=10,000 frames three times at exactly the same PC, USB
   port, ROI, exposure, shutter and AUTO_SEQUENCE trigger. Record measured fps
   and whether timing includes disk saving. Put capture baseline in camware_fps.
2. Close CamWare; run this cell. Script compares capture-only and saved scans.
3. Close script handles (automatic). In GUI repeat 10,000 frames three times,
   timing Start -> finished AND inspecting saved files. Record GUI times.
4. Repeat with normal lab apps open and the intended disk; compare variability.
Benchmarks: zero missing IDs/errors. Proposed saved total rate >=90% of matched
CamWare capture rate; inspect host-read rate too, since close/flush overhead
affects total rate. A failed speed target prompts diagnosis, not frame dropping.
High capture/low save: disk/writer. Low capture and save vs CamWare: SDK/reader.
Good script/slow GUI: display/orchestration. Mock rates measure synthesis cost.


In [ ]:
REPORT_4 = lab.execute(4, CONFIG)


%% 5 — Repeated and sustained reliability
1. Review space estimate: default repeated steps + 5-minute run ~74 GB raw;
   script requires 1.5x headroom. Reduce duration for a preliminary trial only.
2. Watch Task Manager memory/CPU/disk while running. Record start/end and peak
   memory plus any upward trend; repeat at least once after PC warm-up.
3. Use observed_acquisition_span_s: fixed N is only an estimated duration.
   If under 95% of intended time, set target_fps to measured rate and rerun.
4. Separately run a representative GUI multi-step experiment for the intended
   unattended-session duration, subject to available disk space.
Benchmarks: every file valid, zero gaps/overflow/queue errors; no progressive
memory growth after warm-up; no degradation >10% vs section 4. Record actual
maxima rather than inventing a universal RAM or p99 limit. Queue/FIFO failure
is always a failed acquisition even when average fps looks good.
Scan setpoints currently label metadata and DO NOT command a physical stage.


In [ ]:
REPORT_5 = lab.execute(5, CONFIG)


%% 6 — Recovery and GUI usability (automatic fault tests always use a mock)
1. Run this cell: abort + injected writer error retain two incomplete files;
   a subsequent mock acquisition must complete without process restart.
2. GUI: repeat Live -> Stop -> Scan -> Stop -> Live ten times; include rapid
   double-clicks. During a scan try source/ROI/mode changes; controls must safely
   block or serialize them. Resize window; edit colour limits; draw one ROI.
3. Stop a physical short scan. Expect an explicit abort, an incomplete .partial
   for the interrupted step, and successful next scan. Preserve finished steps.
4. MOCK ONLY: choose an unwritable destination and test Start. For low-space,
   use an isolated disposable small test volume; do not fill the research drive.
   Current free-space preflight applies to execute_scan; direct per-step capture
   is protected here by prepare(). Error must be clear; nothing marked complete.
5. With recording STOPPED, disconnect/reconnect camera; verify clear disconnected
   status and successful reconnect. Close app during mock acquisition and reopen.
6. Trigger timeout is deferred to section 8. Do not unplug other lab instruments.
Benchmarks: no GUI crash/deadlock, no simultaneous camera ownership, failure
visible, no incomplete .h5, next run succeeds. Provisional normal UI response
<1 s; Stop finishes within current SDK wait + writer drain. Record seconds;
shutter reboot is a separately expected delay, not a responsiveness failure.


In [ ]:
REPORT_6 = lab.execute(6, CONFIG)


%% 7 — Raw-data integrity, MATLAB interoperability and copy verification
1. Run this cell after short/speed tests, or set data_files to selected .h5 paths.
   Every pixel is read; hashes are retained. This can take time for large files.
2. Copy a file to its analysis/archive destination, rerun with data_files pointing
   to the copy, and require identical pixel_sha256, counts, metadata and settings.
   Pixel hash excludes metadata: compare reported settings and full file SHA256
   separately if requiring a byte-for-byte copy (Get-FileHash in PowerShell).
3. MATLAB, replace f with one reported path:
   f = 'D:\CameraTests\acceptance_short_...h5';
   info = h5info(f, '/images'); disp(info.Dataspace.Size);
   disp(h5readatt(f, '/images', 'dimension_order'));
   roi = double(h5readatt(f, '/', 'roi_bounds'));
   n = double(h5readatt(f, '/', 'frames_written'));
   assert(isequal(double(info.Dataspace.Size(:))', [roi(3)-roi(1), roi(4)-roi(2), n]));
   % Read ONE frame, avoid loading a multi-GB stack:
   I = h5read(f, '/images', [1 1 1], [roi(3)-roi(1), roi(4)-roi(2), 1]);
   assert(isa(I, 'uint16')); imagesc(I'); axis image; colorbar;
   fprintf('mean=%g min=%g max=%g\n', mean(double(I(:))), min(I(:)), max(I(:)));
   % These values must match frame 0 sample_statistics in the JSON report.
4. Acquire labelled dark and stable-illumination short files at identical settings
   using safe existing lab procedures. Review clipping, row artifacts, ROI edges,
   unexpected dark/signal shifts versus matched CamWare files and repeatability.
Benchmarks: all pixels readable, exact counts/axes/settings, Python/MATLAB
sample statistics agree, no display scaling in raw ADU, exact archive hashes.
Saturation/noise acceptance depends on experiment and calibration; document
limits before using these files for correlation measurements.


In [ ]:
REPORT_7 = lab.execute(7, CONFIG)


%% 8 — OPTIONAL / DEFERRED: external trigger and shot accounting (last)
Default trigger_bench_ready=False: this section records DEFERRED and opens no camera.
1. Verify connector, voltage, polarity and pulse width against installed manual.
   Connect a known pulse source and an independent counter/scope. Configure a
   finite train: pulses must occur ONLY while the camera is armed. Use camera
   busy/exposure outputs or verified hardware gating to bound the active window.
2. Start at a known slow rate (e.g. 10 Hz after verifying electrical settings),
   trigger_frames=100; pulse spacing must remain below the driver's ~1 s wait
   timeout. Set trigger_bench_ready=True and source="physical", then run cell.
   Arrange the finite train to start after recording is armed, not before.
3. Record independently observed pulses in the actual acquisition window. Set
   observed_trigger_pulses for the next run; never replace measurement with the
   requested count. A train longer than N cannot be diagnosed merely by stopping
   acquisition after N frames. Check scope busy/exposure and end-of-train timing.
4. Increase rate gradually toward the SCIENTIFIC requirement, repeating counts
   and scope timing. Add no-pulse timeout case: explicit failure, no completed
   .h5, then recovery in AUTO_SEQUENCE. Disable pulse source after each trial.
Benchmarks: independent pulses = exposures = saved frames, correct shutter timing,
zero ID gaps/errors at required rate. Consecutive recorder IDs alone cannot
detect laser pulses the camera never accepted. Host timestamps are receipt times.
Datasheet reference: 455 fps GS at 2560x128 USB; not proof for this ROI or 1 kHz.
https://www.excelitas.com/assets/product/document/pcoedge-55-usb-datasheet.pdf?file=26806.pdf


In [ ]:
REPORT_8 = lab.execute(8, CONFIG)
